In [ ]:
!pip install -q ultralytics
!pip install -q albumentations timm gdown


In [ ]:
import os
import cv2
import math
import random
import warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

import albumentations as A
from albumentations.pytorch import ToTensorV2

from ultralytics import YOLO

warnings.filterwarnings('ignore')
print("Libraries loaded successfully!")


In [ ]:
class CFG:
    # Seed & Environment
    seed = 42
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    num_workers = 4
    use_multi_gpu = torch.cuda.device_count() > 1  # True jika T4x2

    # Paths
    base_dir = "/kaggle/input/datasets/marvelirawan/arsip-doksli-nopal-jomok/arsip-doksli-nopal-jomok"
    train_dir = f"{base_dir}/train"
    test_dir  = f"{base_dir}/test"
    sample_sub_path = f"{base_dir}/samplesubmission.csv"

    # Preprocessing (YOLO Face Cropping)
    crop_size = 448
    margin    = 1        # margin terbaik dari eksperimen sebelumnya

    # PatchNet Config
    patch_size        = 224
    num_patches_train = 5
    num_patches_test  = 9

    # Backbone
    backbone_name = 'dinov2_vitb14_reg'  # Base (embed_dim=768)
    embed_dim     = 768                  # DINOv2-Base
    num_reg_tokens = 4                   # Register tokens DINOv2
    # Fitur gabungan: CLS (768) + mean register tokens (768) = 1536
    fused_dim     = embed_dim * 2

    # Training
    epochs         = 25
    batch_size     = 8   # Per GPU; efektif 16 di T4x2
    lr_backbone    = 5e-6
    lr_head        = 5e-5
    weight_decay   = 1e-4
    n_fold         = 5
    warmup_pct     = 0.1  # 10% epoch pertama untuk warmup (OneCycleLR)
    grad_clip      = 1.0

    # Loss Weights
    lambda_sim     = 0.3
    lambda_supcon  = 0.2
    label_smoothing = 0.1
    temperature_supcon = 0.07

    # Class weights — lebih besar untuk kelas langka / ambigu
    # [realperson, fake_printed, fake_screen, fake_mask, fake_mannequin, fake_unknown]
    class_weights = [1.0, 1.0, 1.0, 1.5, 1.5, 2.0]

    # Classes mapping
    classes = [
        'realperson', 'fake_printed', 'fake_screen',
        'fake_mask', 'fake_mannequin', 'fake_unknown'
    ]
    class_to_idx = {c: i for i, c in enumerate(classes)}
    idx_to_class = {i: c for i, c in enumerate(classes)}

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG.seed)
print(f"Device      : {CFG.device}")
print(f"GPU Count   : {torch.cuda.device_count()}")
print(f"Multi-GPU   : {CFG.use_multi_gpu}")
print(f"Backbone    : {CFG.backbone_name} (embed_dim={CFG.embed_dim})")
print(f"Fused dim   : {CFG.fused_dim}  (CLS + mean register tokens)")


In [ ]:
# Download YOLOv8-face Medium
!gdown --id 1IJZBcyMHGhzAi0G4aZLcqryqZSjPsps- -O yolov8m-face.pt

face_model = YOLO('yolov8m-face.pt')
print("YOLOv8-face loaded!")


In [ ]:
def process_and_crop_face(img_path, model, margin=CFG.margin, output_size=CFG.crop_size):
    img = cv2.imread(img_path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    rotations = [
        (img, 0),
        (cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE), 90),
        (cv2.rotate(img, cv2.ROTATE_180), 180),
        (cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE), 270)
    ]

    best_conf = -1.0
    best_box  = None
    best_img  = None

    for rot_img, angle in rotations:
        results = model(rot_img, verbose=False)[0]
        boxes   = results.boxes.data.cpu().numpy()
        if len(boxes) > 0:
            max_idx = np.argmax(boxes[:, 4])
            conf    = boxes[max_idx, 4]
            if conf > best_conf:
                best_conf = conf
                best_box  = boxes[max_idx, :4]
                best_img  = rot_img

    if best_box is not None:
        x1, y1, x2, y2 = best_box
        h_img, w_img   = best_img.shape[:2]
        w_box = x2 - x1
        h_box = y2 - y1
        x1 = max(0, int(x1 - margin * w_box))
        y1 = max(0, int(y1 - margin * h_box))
        x2 = min(w_img, int(x2 + margin * w_box))
        y2 = min(h_img, int(y2 + margin * h_box))
        crop_img = best_img[y1:y2, x1:x2]
    else:
        crop_img = img  # Fallback: gambar penuh

    final_img = cv2.resize(crop_img, (output_size, output_size))
    return final_img


In [ ]:
out_train_dir = "/kaggle/working/cropped_data/train"
out_test_dir  = "/kaggle/working/cropped_data/test"
os.makedirs(out_train_dir, exist_ok=True)
os.makedirs(out_test_dir,  exist_ok=True)

train_records = []

print("=== Memproses Data Train ===")
for cls_name in CFG.classes:
    cls_dir = os.path.join(CFG.train_dir, cls_name)
    if not os.path.exists(cls_dir):
        continue
    os.makedirs(os.path.join(out_train_dir, cls_name), exist_ok=True)

    for img_name in tqdm(os.listdir(cls_dir), desc=f"Cropping {cls_name}"):
        img_path     = os.path.join(cls_dir, img_name)
        cropped_img  = process_and_crop_face(img_path, face_model)
        if cropped_img is not None:
            save_path = os.path.join(out_train_dir, cls_name, img_name)
            cv2.imwrite(save_path, cv2.cvtColor(cropped_img, cv2.COLOR_RGB2BGR))
            train_records.append({
                'image_path': save_path,
                'label':      cls_name,
                'target':     CFG.class_to_idx[cls_name]
            })

train_df = pd.DataFrame(train_records)
train_df.to_csv("/kaggle/working/train_df.csv", index=False)
print(f"Total Train Images: {len(train_df)}")
print(train_df['label'].value_counts())

print("\n=== Memproses Data Test ===")
test_records    = []
test_img_names  = sorted(os.listdir(CFG.test_dir))

for img_name in tqdm(test_img_names, desc="Cropping Test Set"):
    img_path    = os.path.join(CFG.test_dir, img_name)
    cropped_img = process_and_crop_face(img_path, face_model)
    if cropped_img is not None:
        save_path = os.path.join(out_test_dir, img_name)
        cv2.imwrite(save_path, cv2.cvtColor(cropped_img, cv2.COLOR_RGB2BGR))
        test_records.append({'id': img_name.split('.')[0], 'image_path': save_path})

test_df = pd.DataFrame(test_records)
test_df.to_csv("/kaggle/working/test_df.csv", index=False)
print(f"Total Test Images: {len(test_df)}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import cv2
import os

# Setup visualisasi elegan (Dark Mode menyesuaikan cover kita)
plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(12, 6), facecolor='#0A0F1A')
ax.set_facecolor('#0A0F1A')

# Palet warna khusus: Real = Hijau, Fake = Merah/Ungu/Biru
colors = ['#10B981'] + ['#EF4444', '#F59E0B', '#8B5CF6', '#EC4899', '#3B82F6']
order_classes = ['realperson', 'fake_printed', 'fake_screen', 'fake_mask', 'fake_mannequin', 'fake_unknown']

# Asumsikan variabel dataframe kamu adalah `train_df` (sesuaikan jika namanya berbeda)
sns.countplot(data=train_df, y='label', order=order_classes, palette=colors, ax=ax)

# Anotasi angka
for p in ax.patches:
    ax.annotate(f'{int(p.get_width())}',
                (p.get_width(), p.get_y() + p.get_height() / 2.),
                ha='left', va='center',
                xytext=(5, 0), textcoords='offset points',
                color='white', fontsize=12, fontweight='bold')

ax.set_title('Distribusi Kelas Dataset FAS (Tingkat Imbalance Ekstrem)', fontsize=16, fontweight='bold', color='#38BDF8', pad=20)
ax.set_xlabel('Jumlah Gambar', fontsize=12, color='#94A3B8')
ax.set_ylabel('Tipe Serangan', fontsize=12, color='#94A3B8')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_color('#475569')
ax.spines['left'].set_color('#475569')
ax.tick_params(colors='#CBD5E1')

plt.tight_layout()
plt.show()


In [ ]:
# Kita buat grid 2 baris x 3 kolom untuk menampilkan keenam kelas
fig, axes = plt.subplots(2, 3, figsize=(15, 10), facecolor='#0A0F1A')
fig.suptitle('Verifikasi Potongan Wajah (YOLOv8 Crops)', fontsize=18, color='#38BDF8', fontweight='bold', y=1.02)
axes = axes.flatten()

for i, cls_name in enumerate(order_classes):
    # Ambil 1 sampel acak dari tiap kelas (Sesuaikan nama kolom 'path'/'image_path' dengan df kamu)
    sample_path = train_df[train_df['label'] == cls_name].sample(1, random_state=48)['image_path'].iloc[0]

    # Baca gambar hasil crop
    img = cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB)

    # Plot Local Wajah
    axes[i].imshow(img)
    color_title = '#10B981' if cls_name == 'realperson' else '#EF4444'
    axes[i].set_title(f"Class: {cls_name}", color=color_title, fontsize=14, fontweight='bold')
    axes[i].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
def plot_fourier_spectrum(img_path, ax_img, ax_fft, title):
    # Baca dalam Grayscale untuk analisis tekstur murni
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    # Hitung 2D Fast Fourier Transform (FFT)
    f = np.fft.fft2(img)
    fshift = np.fft.fftshift(f)
    magnitude_spectrum = 20 * np.log(np.abs(fshift) + 1)

    ax_img.imshow(img, cmap='gray')
    ax_img.set_title(title, color='#E2E8F0', fontsize=12)
    ax_img.axis('off')

    ax_fft.imshow(magnitude_spectrum, cmap='magma')
    ax_fft.set_title("FFT Frequency Spectrum", color='#F59E0B', fontsize=12)
    ax_fft.axis('off')

target_classes = ['realperson', 'fake_printed', 'fake_screen']
fig, axes = plt.subplots(3, 2, figsize=(10, 12), facecolor='#0A0F1A')
fig.suptitle('Deteksi Artifact Spoofing via Fourier Transform', fontsize=16, color='#38BDF8', fontweight='bold', y=1.02)

for i, cls_name in enumerate(target_classes):
    sample_path = train_df[train_df['label'] == cls_name].sample(1, random_state=2)['image_path'].iloc[0]
    plot_fourier_spectrum(sample_path, axes[i, 0], axes[i, 1], f"YOLO Crop: {cls_name}")

plt.tight_layout()
plt.show()


In [ ]:
# Augmentasi Training — termasuk simulasi serangan screen, print, dan noise
train_transform = A.Compose([
    # --- Level 1: Spatial & Geometric ---
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(
        shift_limit=0.05, scale_limit=0.1, rotate_limit=15,
        border_mode=cv2.BORDER_CONSTANT, p=0.5
    ),

    # --- Level 2: Simulasi Artefak Serangan ---
    # Simulasi Moiré pattern (screen artifact) dan distorsi kisi
    A.OneOf([
        A.GridDistortion(num_steps=5, distort_limit=0.1, p=1.0),
        A.OpticalDistortion(distort_limit=0.05, shift_limit=0.05, p=1.0),
    ], p=0.3),

    # Simulasi JPEG compression artifact (print & screen attack)
    A.OneOf([
        A.ImageCompression(quality_lower=30, quality_upper=80, p=1.0),
        A.Downscale(scale_min=0.5, scale_max=0.9, p=1.0),
    ], p=0.45),

    # Simulasi noise kamera dan sensor
    A.OneOf([
        A.GaussNoise(var_limit=(10.0, 60.0), p=1.0),
        A.ISONoise(color_shift=(0.01, 0.06), intensity=(0.1, 0.5), p=1.0),
        A.MultiplicativeNoise(multiplier=(0.9, 1.1), p=1.0),
    ], p=0.4),

    # Simulasi blur/sharpness (defocus atau layar)
    A.OneOf([
        A.Blur(blur_limit=5, p=1.0),
        A.GaussianBlur(blur_limit=(3, 7), p=1.0),
        A.Sharpen(alpha=(0.1, 0.4), lightness=(0.5, 1.0), p=1.0),
    ], p=0.35),

    # --- Level 3: Variasi Warna & Pencahayaan ---
    A.ColorJitter(
        brightness=0.25, contrast=0.25,
        saturation=0.35, hue=0.1, p=0.5
    ),
    A.HueSaturationValue(
        hue_shift_limit=12, sat_shift_limit=25, val_shift_limit=12, p=0.4
    ),
    A.RandomBrightnessContrast(
        brightness_limit=0.2, contrast_limit=0.2, p=0.5
    ),

    # Simulasi pencahayaan tidak merata (overexposure / shadow)
    A.OneOf([
        A.RandomShadow(shadow_roi=(0, 0.5, 1, 1), p=1.0),
        A.RandomToneCurve(scale=0.15, p=1.0),
    ], p=0.2),

    # --- Standarisasi ImageNet (DINOv2) ---
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# Augmentasi Validasi / Test — hanya normalisasi
valid_transform = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

print("Augmentasi bertingkat (anti-spoofing) siap!")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  AUGMENTATION VISUALIZATION  —  Level 2 (Anti-Spoofing Artifact Simulation)
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot   as plt
import matplotlib.patches  as mpatches
import albumentations      as A
import cv2, random
import numpy as np

# ── Definisi tiap Level 2 augmentation secara TERPISAH (p=1.0 agar selalu aktif)
# Tidak pakai normalisasi/ToTensorV2 supaya bisa di-display langsung ──────────

aug_configs = [
    {
        'name':    'Original',
        'tag':     'RAW',
        'color':   '#38BDF8',
        'desc':    'Gambar hasil\nYOLO crop',
        'transform': A.Compose([])   # identity
    },
    {
        'name':    'Moiré Simulation\n(GridDistortion)',
        'tag':     'MOIRÉ',
        'color':   '#F59E0B',
        'desc':    'Meniru pola gelombang\npada layar digital',
        'transform': A.Compose([
            A.GridDistortion(
                num_steps=7, distort_limit=0.45,
                interpolation=cv2.INTER_LINEAR, p=1.0
            ),
            A.OpticalDistortion(
                distort_limit=0.25, shift_limit=0.15, p=1.0
            ),
        ])
    },
    {
        'name':    'JPEG Artifact\n(Compression)',
        'tag':     'JPEG',
        'color':   '#8B5CF6',
        'desc':    'Meniru hasil cetak\nberkualitas rendah',
        'transform': A.Compose([
            A.ImageCompression(quality_lower=5, quality_upper=20, p=1.0),
            A.Downscale(scale_min=0.25, scale_max=0.4, p=1.0),
        ])
    },
    {
        'name':    'Sensor Noise\n(GaussNoise + ISO)',
        'tag':     'NOISE',
        'color':   '#EC4899',
        'desc':    'Meniru grain sensor\nkondisi low-light',
        'transform': A.Compose([
            A.GaussNoise(var_limit=(80.0, 160.0), p=0.6),
            A.ISONoise(
                color_shift=(0.08, 0.18),
                intensity=(0.5, 0.9), p=1.0
            ),
            A.MultiplicativeNoise(multiplier=(0.75, 1.25), p=1.0),
        ])
    },
    {
        'name':    'Defocus Blur\n(GaussianBlur)',
        'tag':     'BLUR',
        'color':   '#10B981',
        'desc':    'Meniru kamera\nkehilangan fokus',
        'transform': A.Compose([
            A.GaussianBlur(blur_limit=(15, 21), p=1.0),
        ])
    },
]

N_AUGS    = len(aug_configs)    # 5 kolom

# ── Pilih 2 kelas representatif ───────────────────────────────────────────────
SAMPLE_CLASSES = ['realperson', 'fake_screen']
RAND_SEED      = 21

# ── Warna background per kelas ────────────────────────────────────────────────
CLASS_COLORS = {
    'realperson':  '#10B981',
    'fake_screen': '#3B82F6',
}

BG_DARK  = '#0A0F1A'
BG_PANEL = '#080D18'

plt.style.use('dark_background')

fig, axes = plt.subplots(
    len(SAMPLE_CLASSES), N_AUGS,
    figsize=(N_AUGS * 3.2, len(SAMPLE_CLASSES) * 3.6),
    facecolor=BG_DARK,
    gridspec_kw={'wspace': 0.06, 'hspace': 0.38}
)

random.seed(RAND_SEED)
np.random.seed(RAND_SEED)

for row_idx, cls_name in enumerate(SAMPLE_CLASSES):
    # ── Ambil 1 sampel acak dari kelas ini ────────────────────────────────────
    sample_path = (
        train_df[train_df['label'] == cls_name]
        .sample(1, random_state=RAND_SEED)['image_path']
        .iloc[0]
    )
    img_orig = cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB)

    cls_color = CLASS_COLORS.get(cls_name, '#94A3B8')

    for col_idx, aug_cfg in enumerate(aug_configs):
        ax = axes[row_idx][col_idx]

        # ── Terapkan augmentasi ───────────────────────────────────────────────
        augmented = aug_cfg['transform'](image=img_orig.copy())['image']

        # ── Plot gambar ───────────────────────────────────────────────────────
        ax.imshow(augmented, aspect='auto')
        ax.set_xticks([]); ax.set_yticks([])

        # ── Border styling ────────────────────────────────────────────────────
        edge_color = cls_color if col_idx == 0 else aug_cfg['color']
        edge_width = 2.5       if col_idx == 0 else 1.8
        for spine in ax.spines.values():
            spine.set_edgecolor(edge_color)
            spine.set_linewidth(edge_width)

        # ── Tag badge (kiri atas) ─────────────────────────────────────────────
        tag_bg = cls_color if col_idx == 0 else aug_cfg['color']
        ax.text(
            0.04, 0.97, aug_cfg['tag'],
            transform=ax.transAxes,
            ha='left', va='top',
            fontsize=7.5, fontweight='bold', color='#0A0F1A',
            bbox=dict(
                boxstyle='round,pad=0.28',
                facecolor=tag_bg,
                edgecolor='none',
                alpha=0.92
            )
        )

        # ── Judul kolom (hanya baris pertama) ────────────────────────────────
        if row_idx == 0:
            ax.set_title(
                aug_cfg['name'],
                color=aug_cfg['color'] if col_idx > 0 else '#F1F5F9',
                fontsize=9,
                fontweight='bold' if col_idx == 0 else 'normal',
                pad=7
            )

        # ── Label kelas (hanya kolom pertama / original) ─────────────────────
        if col_idx == 0:
            ax.set_ylabel(
                cls_name,
                color=cls_color, fontsize=10,
                fontweight='bold', labelpad=8
            )

        # ── Deskripsi singkat di bawah (hanya baris terakhir) ─────────────────
        if row_idx == len(SAMPLE_CLASSES) - 1:
            ax.text(
                0.5, -0.08,
                aug_cfg['desc'],
                transform=ax.transAxes,
                ha='center', va='top',
                fontsize=7.5, color='#475569',
                linespacing=1.4
            )

# ── Panel separator garis vertikal (setelah kolom Original) ──────────────────
# Menggunakan figure line via figline trick
sep_x = (1 / N_AUGS) + 0.003
line = plt.Line2D(
    [sep_x, sep_x], [0.06, 0.94],
    transform=fig.transFigure,
    color='#334155', lw=1.0, ls='--', alpha=0.7
)
fig.add_artist(line)

# ── Label kolom kategori ──────────────────────────────────────────────────────
fig.text(
    (1 / N_AUGS) / 2, 0.97,
    'INPUT', ha='center', va='bottom',
    color='#94A3B8', fontsize=8.5, fontweight='bold',
    transform=fig.transFigure
)
fig.text(
    (1 / N_AUGS) + (1 - 1/N_AUGS) / 2, 0.97,
    '← Level 2 Augmentations (Anti-Spoofing Artifact Simulation) →',
    ha='center', va='bottom',
    color='#F59E0B', fontsize=8.5, fontweight='bold',
    transform=fig.transFigure
)

# ── Super Title ───────────────────────────────────────────────────────────────
fig.suptitle(
    'Anti-Spoofing Augmentation  ·  Level 2: Serangan Artefak Digital & Fisik',
    color='#38BDF8', fontsize=13, fontweight='bold',
    y=1.02
)
plt.show()
print("✅ Plot tersimpan: augmentation_level2.png")


In [ ]:
class LivenessPatchDataset(Dataset):
    def __init__(
        self, df, transform=None, mode='train',
        num_patches=CFG.num_patches_train,
        patch_size=CFG.patch_size
    ):
        self.df          = df
        self.transform   = transform
        self.mode        = mode
        self.num_patches = num_patches
        self.patch_size  = patch_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        img     = cv2.imread(row['image_path'])
        img     = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape
        patches  = []

        if self.mode == 'train':
            for _ in range(self.num_patches):
                y     = random.randint(0, h - self.patch_size)
                x     = random.randint(0, w - self.patch_size)
                patch = img[y:y+self.patch_size, x:x+self.patch_size]
                if self.transform:
                    patch = self.transform(image=patch)['image']
                patches.append(patch)
        else:
            # Grid 3×3 = 9 patches
            step_y   = (h - self.patch_size) // 2
            step_x   = (w - self.patch_size) // 2
            y_coords = [0, step_y, h - self.patch_size]
            x_coords = [0, step_x, w - self.patch_size]
            for y in y_coords:
                for x in x_coords:
                    patch = img[y:y+self.patch_size, x:x+self.patch_size]
                    if self.transform:
                        patch = self.transform(image=patch)['image']
                    patches.append(patch)

        patches_tensor = torch.stack(patches)  # [N, C, H, W]

        if self.mode == 'test':
            return patches_tensor, row['id']
        return patches_tensor, torch.tensor(row['target'], dtype=torch.long)

def get_dataloaders(df, fold):
    df_train = df[df['fold'] != fold].reset_index(drop=True)
    df_valid = df[df['fold'] == fold].reset_index(drop=True)

    train_ds = LivenessPatchDataset(df_train, transform=train_transform, mode='train')
    valid_ds = LivenessPatchDataset(df_valid, transform=valid_transform, mode='valid')

    train_loader = DataLoader(
        train_ds, batch_size=CFG.batch_size, shuffle=True,
        num_workers=CFG.num_workers, pin_memory=True, drop_last=True
    )
    valid_loader = DataLoader(
        valid_ds, batch_size=CFG.batch_size, shuffle=False,
        num_workers=CFG.num_workers, pin_memory=True
    )
    return train_loader, valid_loader

# --- Sanity Check ---
train_df = pd.read_csv("/kaggle/working/train_df.csv")
test_df  = pd.read_csv("/kaggle/working/test_df.csv")

skf = StratifiedKFold(n_splits=CFG.n_fold, shuffle=True, random_state=CFG.seed)
train_df['fold'] = -1
for fold, (_, val_idx) in enumerate(skf.split(X=train_df, y=train_df['target'])):
    train_df.loc[val_idx, 'fold'] = fold

_tl, _vl = get_dataloaders(train_df, fold=0)
_p, _l   = next(iter(_tl))
print(f"Train batch shape : {_p.shape}  → (B, N_patches, C, H, W)")
print(f"Label batch shape : {_l.shape}")
print("Dataloader siap!")


In [ ]:
class AttentionPooling(nn.Module):
    """
    Multi-Head Attention Pooling:
    Belajar patch mana yang paling informatif sebagai tanda spoofing,
    bukan sekadar rata-ratakan semua patch secara merata.
    """
    def __init__(self, embed_dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.attn  = nn.MultiheadAttention(
            embed_dim, num_heads=num_heads,
            dropout=dropout, batch_first=True
        )
        self.query = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.norm  = nn.LayerNorm(embed_dim)

    def forward(self, x):  # x: [B, N, D]
        q       = self.query.expand(x.size(0), -1, -1)  # [B, 1, D]
        out, _  = self.attn(q, x, x)
        return self.norm(out.squeeze(1))                 # [B, D]

class PatchDINOv2(nn.Module):
    def __init__(
        self,
        num_classes = len(CFG.classes),
        embed_dim   = CFG.embed_dim,
        fused_dim   = CFG.fused_dim,
        dropout     = 0.3
    ):
        super().__init__()

        # Backbone DINOv2-Base dengan register tokens
        self.backbone = torch.hub.load(
            'facebookresearch/dinov2', CFG.backbone_name
        )

        # Attention Pooling untuk fitur CLS patch
        self.cls_pool = AttentionPooling(embed_dim, num_heads=8, dropout=dropout)

        # Attention Pooling untuk register token
        self.reg_pool = AttentionPooling(embed_dim, num_heads=8, dropout=dropout)

        # Projection sebelum fusion (opsional — bantu stabilisasi gradien)
        self.cls_proj = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim),
            nn.GELU()
        )
        self.reg_proj = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim),
            nn.GELU()
        )

        # Classifier Head
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(fused_dim, fused_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fused_dim // 2, num_classes)
        )

    def forward(self, x):
        # x: [B, N_patches, C, H, W]
        B, N, C, H, W = x.shape
        x_flat = x.view(B * N, C, H, W)

        # Ekstrak CLS token + register tokens dari DINOv2
        out        = self.backbone.forward_features(x_flat)
        cls_tokens = out['x_norm_clstoken']   # [B*N, embed_dim]
        reg_tokens = out['x_norm_regtokens']  # [B*N, num_reg, embed_dim]

        # Bentuk ulang ke [B, N, embed_dim]
        cls_tokens = cls_tokens.reshape(B, N, -1)
        # [B*N, num_reg, embed_dim] → [B, N*num_reg, embed_dim]
        num_reg    = reg_tokens.shape[1]
        reg_tokens = reg_tokens.reshape(B, N * num_reg, -1)

        # Attention Pooling: rangkum semua patch menjadi satu vektor
        cls_agg = self.cls_pool(cls_tokens)  # [B, embed_dim]
        reg_agg = self.reg_pool(reg_tokens)  # [B, embed_dim]

        # Proyeksi & Fusi
        cls_agg = self.cls_proj(cls_agg)
        reg_agg = self.reg_proj(reg_agg)
        fused   = torch.cat([cls_agg, reg_agg], dim=-1)  # [B, fused_dim]

        logits = self.head(fused)

        # Kembalikan logits + patch CLS embeddings untuk loss tambahan
        return logits, cls_tokens  # cls_tokens: [B, N, embed_dim]

# Test bentuk
print("Mengunduh DINOv2-Base...")
_model  = PatchDINOv2().to(CFG.device)
_input  = _p.to(CFG.device)
_logits, _embeds = _model(_input)
print(f"Logits shape         : {_logits.shape}  → (B, num_classes)")
print(f"Patch CLS embeds     : {_embeds.shape}  → (B, N_patches, embed_dim)")
del _model, _input, _logits, _embeds
torch.cuda.empty_cache()
print("Arsitektur PatchDINOv2 + AttentionPooling + RegisterTokens siap!")


In [ ]:
class SupervisedContrastiveLoss(nn.Module):
    """
    SupCon Loss — memaksa embedding kelas yang sama berdekatan
    dan kelas berbeda berjauhan di ruang fitur.
    Sangat membantu untuk kelas ambigu seperti fake_unknown.
    """
    def __init__(self, temperature=CFG.temperature_supcon):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        # features: [B, D]  (mean dari patch embeddings, sudah di-normalize)
        features = F.normalize(features, dim=1)
        sim_mat  = torch.matmul(features, features.T) / self.temperature  # [B, B]

        labels   = labels.view(-1, 1)
        mask     = torch.eq(labels, labels.T).float().to(features.device)
        mask.fill_diagonal_(0)  # Exclude diri sendiri

        exp_sim  = torch.exp(sim_mat)
        # Untuk setiap anchor, hitung log-softmax terhadap semua sample lain
        log_prob = sim_mat - torch.log(
            exp_sim.sum(dim=1, keepdim=True) - exp_sim.diagonal().unsqueeze(1)
        )
        n_pos    = mask.sum(dim=1).clamp(min=1)
        loss     = -(mask * log_prob).sum(dim=1) / n_pos
        return loss.mean()

class PatchNetLoss(nn.Module):
    def __init__(
        self,
        num_classes    = len(CFG.classes),
        scale          = 30.0,
        lambda_sim     = CFG.lambda_sim,
        lambda_supcon  = CFG.lambda_supcon
    ):
        super().__init__()
        self.scale          = scale
        self.lambda_sim     = lambda_sim
        self.lambda_supcon  = lambda_supcon

        # Asymmetric margin per kelas
        # [realperson, fake_printed, fake_screen, fake_mask, fake_mannequin, fake_unknown]
        self.margins = torch.tensor([0.4, 0.2, 0.2, 0.2, 0.2, 0.1])

        # Class-weighted CE dengan label smoothing
        class_weights = torch.tensor(CFG.class_weights)
        self.ce_loss  = nn.CrossEntropyLoss(
            weight         = class_weights,
            label_smoothing = CFG.label_smoothing
        )

        self.supcon = SupervisedContrastiveLoss()

    def to(self, device):
        super().to(device)
        self.margins = self.margins.to(device)
        self.ce_loss.weight = self.ce_loss.weight.to(device)
        return self

    def forward(self, logits, patch_embeddings, labels):
        """
        logits           : [B, num_classes]
        patch_embeddings : [B, N_patches, embed_dim]
        labels           : [B]
        """
        # 1. Asymmetric AM-Softmax (AAMS)
        m              = self.margins[labels]  # [B]
        cosine         = F.normalize(logits, p=2, dim=1)
        target_oh      = torch.zeros_like(cosine)
        target_oh.scatter_(1, labels.view(-1, 1), 1.0)
        cosine_m       = cosine - m.view(-1, 1) * target_oh
        scaled_logits  = cosine_m * self.scale
        L_AAMS         = self.ce_loss(scaled_logits, labels)

        # 2. Similarity Loss — konsistensi antar patch dalam satu gambar
        mean_emb = patch_embeddings.mean(dim=1, keepdim=True)
        L_Sim    = F.mse_loss(patch_embeddings, mean_emb.expand_as(patch_embeddings))

        # 3. Supervised Contrastive Loss — fitur kelas sama berdekatan
        mean_feat = patch_embeddings.mean(dim=1)  # [B, embed_dim]
        L_SupCon  = self.supcon(mean_feat, labels)

        # 4. Total Loss
        total = L_AAMS + (self.lambda_sim * L_Sim) + (self.lambda_supcon * L_SupCon)
        return total, L_AAMS, L_Sim, L_SupCon

print("Loss function (AAMS + Similarity + SupCon) siap!")


In [ ]:
import time
import gc

def train_epoch(model, dataloader, criterion, optimizer, scaler, scheduler, epoch, device):
    model.train()
    total_loss = 0.0

    # Freeze backbone 4 epoch pertama (warmup head saja)
    backbone = model.module.backbone if hasattr(model, 'module') else model.backbone
    if epoch < 4:
        for p in backbone.parameters():
            p.requires_grad = False
    else:
        for p in backbone.parameters():
            p.requires_grad = True

    pbar = tqdm(dataloader, desc=f"Train Epoch {epoch+1}/{CFG.epochs}", leave=False)
    for patches, labels in pbar:
        patches, labels = patches.to(device), labels.to(device)
        optimizer.zero_grad()

        with autocast():
            logits, patch_embeds      = model(patches)
            loss, l_aams, l_sim, l_sc = criterion(logits, patch_embeds, labels)

        scaler.scale(loss).backward()
        # Gradient clipping untuk stabilitas
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
        scaler.step(optimizer)
        scaler.update()

        # OneCycleLR step per batch
        scheduler.step()

        total_loss += loss.item()
        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            aams=f"{l_aams.item():.3f}",
            sim=f"{l_sim.item():.3f}",
            sc=f"{l_sc.item():.3f}"
        )

    return total_loss / len(dataloader)

@torch.no_grad()
def valid_epoch(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds  = []
    all_labels = []

    pbar = tqdm(dataloader, desc="Validating", leave=False)
    for patches, labels in pbar:
        patches, labels = patches.to(device), labels.to(device)
        with autocast():
            logits, patch_embeds      = model(patches)
            loss, _, _, _             = criterion(logits, patch_embeds, labels)

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return total_loss / len(dataloader), macro_f1

print("Training & Validation loop siap!")


In [ ]:
def run_training():
    os.makedirs("/kaggle/working/models", exist_ok=True)
    oof_f1_scores = []

    global training_history, oof_all_preds, oof_all_labels
    training_history = {}
    oof_all_preds  = []   # ← TAMBAHAN BARU
    oof_all_labels = []   # ← TAMBAHAN BARU

    for fold in range(CFG.n_fold):
        print(f"\n{'='*22} Fold {fold+1}/{CFG.n_fold} {'='*22}")

        train_loader, valid_loader = get_dataloaders(train_df, fold)

        model = PatchDINOv2(num_classes=len(CFG.classes)).to(CFG.device)
        if CFG.use_multi_gpu:
            print(f"  Menggunakan DataParallel pada {torch.cuda.device_count()} GPU")
            model = nn.DataParallel(model)

        criterion = PatchNetLoss().to(CFG.device)

        backbone_params = (
            model.module.backbone.parameters()
            if hasattr(model, 'module') else model.backbone.parameters()
        )
        head_params = (
            list(model.module.cls_pool.parameters()) +
            list(model.module.reg_pool.parameters()) +
            list(model.module.cls_proj.parameters()) +
            list(model.module.reg_proj.parameters()) +
            list(model.module.head.parameters())
        ) if hasattr(model, 'module') else (
            list(model.cls_pool.parameters()) +
            list(model.reg_pool.parameters()) +
            list(model.cls_proj.parameters()) +
            list(model.reg_proj.parameters()) +
            list(model.head.parameters())
        )

        optimizer = optim.AdamW(
            [
                {'params': backbone_params, 'lr': CFG.lr_backbone},
                {'params': head_params,     'lr': CFG.lr_head}
            ],
            weight_decay=CFG.weight_decay
        )
        scheduler = optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=[CFG.lr_backbone, CFG.lr_head],
            steps_per_epoch=len(train_loader),
            epochs=CFG.epochs,
            pct_start=CFG.warmup_pct,
            anneal_strategy='cos'
        )

        scaler     = GradScaler()
        best_f1    = 0.0
        model_path = f"/kaggle/working/models/dinov2b_fold{fold}.pth"

        fold_history = {
            'train_loss': [], 'valid_loss': [],
            'valid_f1':   [], 'best_epoch': 0
        }

        for epoch in range(CFG.epochs):
            t0         = time.time()
            train_loss = train_epoch(
                model, train_loader, criterion,
                optimizer, scaler, scheduler, epoch, CFG.device
            )
            valid_loss, valid_f1 = valid_epoch(
                model, valid_loader, criterion, CFG.device
            )
            elapsed = time.time() - t0

            fold_history['train_loss'].append(train_loss)
            fold_history['valid_loss'].append(valid_loss)
            fold_history['valid_f1'].append(valid_f1)

            print(
                f"  Epoch {epoch+1:02d} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Val Loss: {valid_loss:.4f} | "
                f"Val F1 Macro: {valid_f1:.4f} | "
                f"Time: {elapsed:.0f}s"
            )

            if valid_f1 > best_f1:
                print(f"  ★ F1 naik {best_f1:.4f} → {valid_f1:.4f} — menyimpan model...")
                best_f1 = valid_f1
                fold_history['best_epoch'] = epoch
                state = (
                    model.module.state_dict()
                    if hasattr(model, 'module') else model.state_dict()
                )
                torch.save(state, model_path)

        training_history[fold] = fold_history
        oof_f1_scores.append(best_f1)
        print(f"  ▶ Fold {fold+1} selesai. Best Val F1 Macro: {best_f1:.4f}")

        del model, optimizer, scheduler, train_loader
        torch.cuda.empty_cache()
        gc.collect()

        # ── TAMBAHAN BARU: OOF Inference dengan best model fold ini ──────────
        print(f"  → Mengumpulkan OOF predictions dari best model Fold {fold+1}...")
        oof_model = PatchDINOv2(num_classes=len(CFG.classes)).to(CFG.device)
        oof_model.load_state_dict(
            torch.load(model_path, map_location=CFG.device)
        )
        if CFG.use_multi_gpu:
            oof_model = nn.DataParallel(oof_model)
        oof_model.eval()

        fold_preds  = []
        fold_labels = []
        with torch.no_grad():
            for patches, labels in tqdm(valid_loader, desc=f"  OOF Fold {fold+1}", leave=False):
                patches = patches.to(CFG.device)
                with autocast():
                    logits, _ = oof_model(patches)
                preds = torch.argmax(logits, dim=1).cpu().numpy()
                fold_preds.extend(preds)
                fold_labels.extend(labels.numpy())

        oof_all_preds.extend(fold_preds)
        oof_all_labels.extend(fold_labels)

        del oof_model, valid_loader
        torch.cuda.empty_cache()
        gc.collect()
        # ─────────────────────────────────────────────────────────────────────

    print(f"\n{'='*50}")
    print(f"TRAINING SELESAI!")
    print(f"OOF F1 Macro per Fold : {[round(s, 4) for s in oof_f1_scores]}")
    print(f"Rata-rata CV F1 Macro : {np.mean(oof_f1_scores):.4f}")
    print(f"Total OOF samples     : {len(oof_all_preds)}")
    print(f"{'='*50}")

run_training()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  OOF CONFUSION MATRIX  —  All Folds Aggregated
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot    as plt
import matplotlib.patches   as mpatches
import matplotlib.ticker    as mticker
import numpy                as np
from sklearn.metrics        import confusion_matrix, f1_score, classification_report

# ── Data ─────────────────────────────────────────────────────────────────────
y_true     = np.array(oof_all_labels)
y_pred     = np.array(oof_all_preds)
cm         = confusion_matrix(y_true, y_pred)          # raw counts
cm_norm    = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # row-normalize (recall)
n_classes  = len(CFG.classes)
oof_f1     = f1_score(y_true, y_pred, average='macro')

CLASS_LABELS = [
    'realperson', 'fake\nprinted', 'fake\nscreen',
    'fake\nmask',  'fake\nmanneq.', 'fake\nunknown'
]

# ── Palette ───────────────────────────────────────────────────────────────────
BG_DARK   = '#0A0F1A'
BG_PANEL  = '#0D1424'
ACCENT    = '#38BDF8'
GOOD_CLR  = '#10B981'   # high recall (diagonal)
BAD_CLR   = '#EF4444'   # off-diagonal errors
MID_CLR   = '#1E3A5F'   # near-zero cells

plt.style.use('dark_background')

fig, axes = plt.subplots(
    1, 2,
    figsize=(16, 7),
    facecolor=BG_DARK,
    gridspec_kw={'width_ratios': [1.55, 1], 'wspace': 0.38}
)

# ═══════════════════════════════════════════════════════════════════════════════
#  Panel Kiri — Heatmap Confusion Matrix (Row-Normalized)
# ═══════════════════════════════════════════════════════════════════════════════
ax = axes[0]
ax.set_facecolor(BG_PANEL)

# Custom colormap: dark blue → teal → green (diagonal nice), red for errors
from matplotlib.colors import LinearSegmentedColormap
cmap_diag = LinearSegmentedColormap.from_list(
    'fas_cm',
    ['#0A0F1A', '#0F3460', '#1B6CA8', '#10B981', '#34D399'],
    N=256
)

im = ax.imshow(cm_norm, cmap=cmap_diag, vmin=0, vmax=1, aspect='auto')

# Annotasi setiap sel: persen + (count)
for i in range(n_classes):
    for j in range(n_classes):
        val_pct = cm_norm[i, j]
        val_cnt = cm[i, j]
        is_diag = (i == j)

        # Warna teks: putih terang untuk diag & sel gelap, abu untuk sel terang
        txt_color = '#FFFFFF' if val_pct < 0.75 else '#0A0F1A'

        # Baris diagonal: tampilkan lebih besar
        ax.text(
            j, i,
            f'{val_pct:.0%}\n({val_cnt})',
            ha='center', va='center',
            fontsize=9.5 if is_diag else 8,
            fontweight='bold' if is_diag else 'normal',
            color=txt_color,
            alpha=1.0 if val_cnt > 0 else 0.25
        )

# Kotak highlight diagonal
for k in range(n_classes):
    rect = mpatches.FancyBboxPatch(
        (k - 0.48, k - 0.48), 0.96, 0.96,
        boxstyle='round,pad=0.04',
        linewidth=1.8,
        edgecolor=GOOD_CLR,
        facecolor='none',
        zorder=3
    )
    ax.add_patch(rect)

# Label sumbu
ax.set_xticks(range(n_classes))
ax.set_yticks(range(n_classes))
ax.set_xticklabels(CLASS_LABELS, fontsize=9, color='#CBD5E1')
ax.set_yticklabels(CLASS_LABELS, fontsize=9, color='#CBD5E1')
ax.set_xlabel('Predicted Label', fontsize=11, color='#94A3B8', labelpad=10)
ax.set_ylabel('True Label',      fontsize=11, color='#94A3B8', labelpad=10)
ax.tick_params(length=0)

for spine in ax.spines.values():
    spine.set_edgecolor('#1E293B')

# Colorbar
cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cbar.ax.tick_params(colors='#64748B', labelsize=8)
cbar.set_label('Recall (Row-Normalized)', color='#94A3B8', fontsize=9, labelpad=8)

ax.set_title(
    f'OOF Confusion Matrix  (5-Fold, N={len(y_true)})\n'
    f'F1-Macro: {oof_f1:.4f}',
    color='#F1F5F9', fontsize=12, fontweight='bold', pad=14
)

# ═══════════════════════════════════════════════════════════════════════════════
#  Panel Kanan — Per-Class F1, Precision, Recall (Bar Chart)
# ═══════════════════════════════════════════════════════════════════════════════
ax2 = axes[1]
ax2.set_facecolor(BG_PANEL)

report  = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
classes = CFG.classes

per_f1   = [report[str(i)]['f1-score']  for i in range(n_classes)]
per_prec = [report[str(i)]['precision'] for i in range(n_classes)]
per_rec  = [report[str(i)]['recall']    for i in range(n_classes)]

y_pos   = np.arange(n_classes)
bar_h   = 0.24
offset  = bar_h

# Warna bar berdasarkan nilai F1 (hijau jika tinggi, merah jika rendah)
bar_colors = [
    '#10B981' if v >= 0.7 else '#F59E0B' if v >= 0.4 else '#EF4444'
    for v in per_f1
]

b1 = ax2.barh(y_pos + offset,   per_prec, bar_h, color=ACCENT,      alpha=0.75, label='Precision')
b2 = ax2.barh(y_pos,            per_rec,  bar_h, color='#F472B6',   alpha=0.75, label='Recall')
b3 = ax2.barh(y_pos - offset,   per_f1,   bar_h, color=bar_colors,  alpha=0.90, label='F1-Score')

# Nilai di ujung bar F1
for i, v in enumerate(per_f1):
    ax2.text(
        v + 0.01, y_pos[i] - offset,
        f'{v:.3f}', va='center', ha='left',
        fontsize=8.5, fontweight='bold',
        color='#10B981' if v >= 0.7 else '#F59E0B' if v >= 0.4 else '#EF4444'
    )

# Garis F1-macro reference
ax2.axvline(x=oof_f1, color=ACCENT, lw=1.2, ls='--', alpha=0.6)
ax2.text(
    oof_f1 + 0.01, n_classes - 0.5,
    f'Macro\n{oof_f1:.3f}',
    color=ACCENT, fontsize=7.5, va='top'
)

ax2.set_yticks(y_pos)
ax2.set_yticklabels(
    [c.replace('fake_', 'f.') for c in classes],
    fontsize=9, color='#CBD5E1'
)
ax2.set_xlim(0, 1.18)
ax2.set_xlabel('Score', fontsize=10, color='#94A3B8')
ax2.tick_params(colors='#64748B', labelsize=8, length=0)
ax2.xaxis.set_major_locator(mticker.MultipleLocator(0.2))
ax2.grid(True, axis='x', color='#1E293B', lw=0.8, ls='--')

for spine in ax2.spines.values():
    spine.set_edgecolor('#1E293B')

legend = ax2.legend(
    frameon=True, facecolor='#0F172A',
    edgecolor='#334155', labelcolor='#E2E8F0',
    fontsize=8.5, loc='lower right'
)
ax2.set_title(
    'Per-Class Metrics',
    color='#F1F5F9', fontsize=12, fontweight='bold', pad=14
)

# ── Super Title ───────────────────────────────────────────────────────────────
fig.suptitle(
    'OOF Evaluation  ·  DINOv2 + Attention Pooling  ·  Face Anti-Spoofing',
    color='#38BDF8', fontsize=13, fontweight='bold', y=1.02
)
plt.show()
print(f"✅ Plot tersimpan: oof_confusion_matrix.png")
print(f"\n{classification_report(y_true, y_pred, target_names=CFG.classes, zero_division=0)}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  TRAINING CURVE VISUALIZATION  —  Fold 0
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
from matplotlib.lines   import Line2D

PLOT_FOLD = 0   # ← ganti angka ini untuk melihat fold lain (0–4)

h          = training_history[PLOT_FOLD]
epochs_arr = list(range(1, len(h['train_loss']) + 1))
best_ep    = h['best_epoch'] + 1           # 1-indexed
best_f1_v  = max(h['valid_f1'])

# ── Warna palette (konsisten dengan notebook dark theme) ─────────────────────
C_TRAIN  = '#38BDF8'   # cyan  — train loss
C_VAL    = '#F472B6'   # pink  — val loss
C_F1     = '#A78BFA'   # violet — val F1
C_BEST   = '#10B981'   # emerald — best epoch marker
BG_DARK  = '#0A0F1A'
BG_PANEL = '#0F172A'
GRID_CLR = '#1E293B'

plt.style.use('dark_background')
fig = plt.figure(figsize=(14, 6), facecolor=BG_DARK, dpi=150)
gs  = GridSpec(1, 2, figure=fig, wspace=0.38)

# ─────────────────────────────────────────────────────────────────────────────
#  Panel Kiri — Train Loss vs Val Loss
# ─────────────────────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
ax1.set_facecolor(BG_PANEL)

ax1.plot(epochs_arr, h['train_loss'],
         color=C_TRAIN, lw=2.2, marker='o', ms=4, label='Train Loss')
ax1.plot(epochs_arr, h['valid_loss'],
         color=C_VAL,   lw=2.2, marker='s', ms=4, label='Val Loss')

# Shaded area (confidence band visual)
ax1.fill_between(epochs_arr, h['train_loss'], h['valid_loss'],
                 alpha=0.06, color='white')

# Best epoch vertical line
ax1.axvline(x=best_ep, color=C_BEST, lw=1.4, ls='--', alpha=0.8)
ax1.annotate(
    f' Best\n Epoch {best_ep}',
    xy=(best_ep, min(h['valid_loss'])),
    xytext=(best_ep + 0.4, min(h['valid_loss']) * 1.05),
    color=C_BEST, fontsize=8.5, fontweight='bold'
)

ax1.set_title(f'Loss Curve — Fold {PLOT_FOLD + 1}',
              color='#F1F5F9', fontsize=13, fontweight='bold', pad=12)
ax1.set_xlabel('Epoch', color='#94A3B8', fontsize=10)
ax1.set_ylabel('Loss', color='#94A3B8', fontsize=10)
ax1.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax1.tick_params(colors='#64748B', labelsize=8.5)
ax1.grid(True, color=GRID_CLR, lw=0.8, ls='--')
for spine in ax1.spines.values():
    spine.set_edgecolor('#1E293B')

legend1 = ax1.legend(
    frameon=True, facecolor='#0F172A',
    edgecolor='#334155', labelcolor='#E2E8F0',
    fontsize=9, loc='upper right'
)

# ─────────────────────────────────────────────────────────────────────────────
#  Panel Kanan — Val F1-Macro
# ─────────────────────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1])
ax2.set_facecolor(BG_PANEL)

ax2.plot(epochs_arr, h['valid_f1'],
         color=C_F1, lw=2.2, marker='D', ms=4)

# Area under curve
ax2.fill_between(epochs_arr, h['valid_f1'], alpha=0.12, color=C_F1)

# Best F1 marker
ax2.scatter([best_ep], [best_f1_v],
            color=C_BEST, s=90, zorder=5, edgecolors='white', lw=1.2)
ax2.axvline(x=best_ep, color=C_BEST, lw=1.4, ls='--', alpha=0.8)
ax2.annotate(
    f' Best F1\n {best_f1_v:.4f}',
    xy=(best_ep, best_f1_v),
    xytext=(best_ep + 0.4, best_f1_v - 0.02),
    color=C_BEST, fontsize=8.5, fontweight='bold'
)

# Horizontal reference line
ax2.axhline(y=best_f1_v, color=C_BEST, lw=0.8, ls=':', alpha=0.5)

ax2.set_title(f'Val F1-Macro — Fold {PLOT_FOLD + 1}',
              color='#F1F5F9', fontsize=13, fontweight='bold', pad=12)
ax2.set_xlabel('Epoch', color='#94A3B8', fontsize=10)
ax2.set_ylabel('F1-Score (Macro)', color='#94A3B8', fontsize=10)
ax2.set_ylim(0, 1.05)
ax2.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax2.tick_params(colors='#64748B', labelsize=8.5)
ax2.grid(True, color=GRID_CLR, lw=0.8, ls='--')
for spine in ax2.spines.values():
    spine.set_edgecolor('#1E293B')

# ─────────────────────────────────────────────────────────────────────────────
#  Super Title
# ─────────────────────────────────────────────────────────────────────────────
fig.suptitle(
    f'Training Monitoring  ·  DINOv2 + Attention Pooling  ·  Fold {PLOT_FOLD + 1}/{CFG.n_fold}',
    color='#38BDF8', fontsize=14, fontweight='bold', y=1.02
)

plt.tight_layout()
plt.show()
print(f"✅ Plot tersimpan: training_curve_fold{PLOT_FOLD + 1}.png")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  ERROR ANALYSIS  —  3 Random Misclassified OOF Samples
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot    as plt
import matplotlib.patches   as mpatches
import matplotlib.gridspec  as gridspec
import cv2, random
import numpy as np

# ── Rekonstruksi OOF DataFrame (urutan fold → urutan pred) ───────────────────
# valid_loader pakai shuffle=False, jadi urutan pred = urutan baris per fold
oof_parts = [
    train_df[train_df['fold'] == f].reset_index(drop=True)
    for f in range(CFG.n_fold)
]
oof_df = pd.concat(oof_parts, ignore_index=True).copy()
oof_df['pred_idx']   = oof_all_preds
oof_df['pred_label'] = [CFG.idx_to_class[p] for p in oof_all_preds]

# ── Isolasi sample yang salah prediksi ───────────────────────────────────────
error_df = oof_df[oof_df['target'] != oof_df['pred_idx']].reset_index(drop=True)
print(f"Total misclassified samples : {len(error_df)} / {len(oof_df)}")
print(f"Error rate                  : {len(error_df)/len(oof_df)*100:.1f}%\n")
print("Distribusi error per kelas (True Label):")
print(error_df['label'].value_counts().to_string())

# ── Ambil 3 sampel random ─────────────────────────────────────────────────────
N_SAMPLES  = 3
RAND_SEED  = 7      # ← ganti untuk mendapat sampel berbeda
sample_df  = error_df.sample(n=min(N_SAMPLES, len(error_df)), random_state=RAND_SEED)

# ── Warna per kelas ───────────────────────────────────────────────────────────
CLASS_COLORS = {
    'realperson':      '#10B981',
    'fake_printed':    '#F59E0B',
    'fake_screen':     '#3B82F6',
    'fake_mask':       '#8B5CF6',
    'fake_mannequin':  '#EC4899',
    'fake_unknown':    '#EF4444',
}
BG_DARK  = '#0A0F1A'
BG_PANEL = '#0D1424'

plt.style.use('dark_background')
fig = plt.figure(figsize=(15, 7), facecolor=BG_DARK)

# Grid: 1 baris header info + 1 baris gambar, per kolom = 1 sampel
outer = gridspec.GridSpec(
    1, N_SAMPLES,
    figure=fig,
    wspace=0.06,
    left=0.03, right=0.97, top=0.82, bottom=0.04
)

for col_idx, (_, row) in enumerate(sample_df.iterrows()):
    true_lbl = row['label']
    pred_lbl = row['pred_label']
    fold_n   = int(row['fold']) + 1

    # ── Baca gambar crop ──────────────────────────────────────────────────────
    img_bgr = cv2.imread(row['image_path'])
    if img_bgr is None:
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # ── Subplot per sampel ────────────────────────────────────────────────────
    ax = fig.add_subplot(outer[col_idx])
    ax.imshow(img_rgb, aspect='auto')
    ax.set_xticks([]); ax.set_yticks([])

    # Border warna merah (salah prediksi)
    for spine in ax.spines.values():
        spine.set_edgecolor('#EF4444')
        spine.set_linewidth(2.5)

    # ── Badge TRUE LABEL (di atas gambar, luar axes) ──────────────────────────
    true_color = CLASS_COLORS.get(true_lbl, '#94A3B8')
    pred_color = CLASS_COLORS.get(pred_lbl, '#94A3B8')

    ax.set_title(
        f'Sample #{col_idx + 1}  ·  Fold {fold_n}',
        color='#94A3B8', fontsize=9, pad=6, loc='center'
    )

    # ── Label overlay di BAWAH gambar (sebagai xlabel) ───────────────────────
    # True label (hijau/warna kelas)
    ax.text(
        0.5, -0.03,
        '✦  TRUE',
        transform=ax.transAxes,
        ha='center', va='top',
        color='#64748B', fontsize=8, fontweight='bold'
    )
    ax.text(
        0.5, -0.10,
        true_lbl,
        transform=ax.transAxes,
        ha='center', va='top',
        color=true_color, fontsize=11, fontweight='bold'
    )

    # Arrow / separator
    ax.text(
        0.5, -0.19,
        '↓  predicted as',
        transform=ax.transAxes,
        ha='center', va='top',
        color='#EF4444', fontsize=8, style='italic'
    )

    # Predicted label (merah karena salah)
    ax.text(
        0.5, -0.27,
        pred_lbl,
        transform=ax.transAxes,
        ha='center', va='top',
        color=pred_color, fontsize=11, fontweight='bold',
        bbox=dict(
            boxstyle='round,pad=0.35',
            facecolor='#1E0A0A',
            edgecolor=pred_color,
            linewidth=1.4,
            alpha=0.85
        )
    )

# ── Super Title ───────────────────────────────────────────────────────────────
fig.text(
    0.5, 0.97,
    'Error Analysis  ·  Misclassified OOF Samples  ·  DINOv2 + Attention Pooling',
    ha='center', va='top',
    color='#38BDF8', fontsize=13, fontweight='bold'
)
fig.text(
    0.5, 0.91,
    f'Total error: {len(error_df)}/{len(oof_df)} samples  '
    f'({len(error_df)/len(oof_df)*100:.1f}%)  '
    f'·  Menampilkan {N_SAMPLES} sampel acak  '
    f'·  Border merah = prediksi salah',
    ha='center', va='top',
    color='#64748B', fontsize=9
)
plt.show()
print(f"✅ Plot tersimpan: error_analysis.png")


In [ ]:
def run_inference_and_submit():
    print("=== Memulai Fase Inferensi & Ensembling ===")

    test_df   = pd.read_csv("/kaggle/working/test_df.csv")
    test_ds   = LivenessPatchDataset(
        test_df, transform=valid_transform,
        mode='test', num_patches=CFG.num_patches_test
    )
    test_loader = DataLoader(
        test_ds, batch_size=CFG.batch_size, shuffle=False,
        num_workers=CFG.num_workers, pin_memory=True
    )

    # Akumulasi probabilitas per gambar
    ensemble_probs = {
        row['id']: np.zeros(len(CFG.classes))
        for _, row in test_df.iterrows()
    }
    n_loaded = 0

    for fold in range(CFG.n_fold):
        model_path = f"/kaggle/working/models/dinov2b_fold{fold}.pth"
        if not os.path.exists(model_path):
            print(f"  ⚠ Model Fold {fold} tidak ditemukan, skip.")
            continue

        print(f"Memuat Model Fold {fold}...")
        model = PatchDINOv2(num_classes=len(CFG.classes)).to(CFG.device)
        model.load_state_dict(torch.load(model_path, map_location=CFG.device))

        if CFG.use_multi_gpu:
            model = nn.DataParallel(model)
        model.eval()
        n_loaded += 1

        with torch.no_grad():
            for patches, img_ids in tqdm(test_loader, desc=f"Inferensi Fold {fold}"):
                patches = patches.to(CFG.device)
                with autocast():
                    logits, _ = model(patches)
                probs = F.softmax(logits, dim=1).cpu().numpy()
                for i, img_id in enumerate(img_ids):
                    ensemble_probs[img_id] += probs[i]

        del model
        torch.cuda.empty_cache()

    # Rata-ratakan akumulasi probabilitas
    print(f"\nTotal model dimuat: {n_loaded}/{CFG.n_fold}")
    print("=== Membuat File Submission ===")

    records = []
    for img_id, probs in ensemble_probs.items():
        pred_idx   = np.argmax(probs / max(n_loaded, 1))
        pred_label = CFG.idx_to_class[pred_idx]
        records.append({'id': img_id, 'label': pred_label})

    sub_df = (
        pd.DataFrame(records)
        .sort_values('id')
        .reset_index(drop=True)
    )
    sub_df.to_csv("submission.csv", index=False)
    print("✅ 'submission.csv' berhasil dibuat!")
    print(sub_df['label'].value_counts())
    display(sub_df.head(10))

run_inference_and_submit()
